In [ ]:
# Install Required Libraries (run if needed)
!pip install datasets transformers torch scikit-learn

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm
import numpy as np
import json
from pathlib import Path

# Constants
MODEL_NAME = "emrecan/bert-base-turkish-cased-allnli_tr"  # Change to your fine-tuned model if needed
RESULTS_DIR = "results"
LABELS = ["entailment", "neutral", "contradiction"]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
model.to(DEVICE)
model.eval()

config.json:   0%|          | 0.00/910 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/373 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/443M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: emrecan/bert-base-turkish-cased-allnli_tr
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(32000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [ ]:
# Load datasets
snli_ds = load_dataset("yilmazzey/sdp2-nli", "snli_tr_1_1")
multinli_ds = load_dataset("yilmazzey/sdp2-nli", "multinli_tr_1_1")
trglue_ds = load_dataset("yilmazzey/sdp2-nli", "trglue_mnli")

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

In [ ]:
def evaluate_nli(model, tokenizer, ds, dataset_name, split_name="test", num_wrong=None):
    # Get predictions
    premises = ds[split_name]["premise"]
    hypotheses = ds[split_name]["hypothesis"]
    labels = ds[split_name]["label"]

    predictions = []
    with torch.no_grad():
        for p, h in tqdm(zip(premises, hypotheses), total=len(premises)):
            inputs = tokenizer(p, h, truncation=True, padding=True, return_tensors="pt").to(DEVICE)
            outputs = model(**inputs)
            pred = torch.argmax(outputs.logits, dim=1).cpu().item()
            predictions.append(pred)

    predictions = np.array(predictions)
    accuracy = accuracy_score(labels, predictions)
    f1_macro = f1_score(labels, predictions, average="macro")

    # Collect wrong examples
    wrong_examples = []
    for i in range(len(labels)):
        if labels[i] != predictions[i]:
            wrong_examples.append({
                "index": i,
                "premise": premises[i],
                "hypothesis": hypotheses[i],
                "true_label": LABELS[labels[i]],
                "predicted_label": LABELS[predictions[i]],
            })

    # Limit to num_wrong if specified
    if num_wrong is not None:
        wrong_examples = wrong_examples[:num_wrong]

    # Save wrong examples to JSON
    Path(RESULTS_DIR).mkdir(exist_ok=True)
    wrong_path = Path(RESULTS_DIR) / f"wrong_examples_{dataset_name}_{split_name}.json"
    with open(wrong_path, "w") as f:
        json.dump(wrong_examples, f, indent=4, ensure_ascii=False)

    print(f"Saved {len(wrong_examples)} wrong examples to {wrong_path}")

    return accuracy, f1_macro

In [13]:
# Evaluate on all datasets (full splits)
results = {}



# TrGLUE-MNLI: has test_matched and test_mismatched
results['TrGLUE-MNLI (matched)'] = evaluate_nli(model, tokenizer, trglue_ds, "TrGLUE-MNLI", split_name='test_matched', num_wrong=200)  # Example: limit to 50 wrong instances
results['TrGLUE-MNLI (mismatched)'] = evaluate_nli(model, tokenizer, trglue_ds, "TrGLUE-MNLI", split_name='test_mismatched', num_wrong=200)  # Adjust num_wrong as needed

Streaming output truncated to the last 5000 lines.
100%|██████████| 9008/9008 [28:25<00:00,  5.28it/s]


Saved 200 wrong examples to results/wrong_examples_TrGLUE-MNLI_test_matched.json


100%|██████████| 9217/9217 [27:22<00:00,  5.61it/s]


Saved 200 wrong examples to results/wrong_examples_TrGLUE-MNLI_test_mismatched.json
